In [1]:
# ==========================================\n
# CELL 1: IMPORT THƯ VIỆN & CỐ ĐỊNH SEED
# ==========================================\n
import os
import time
import math
import random
import psutil
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from thop import profile
from tqdm.auto import tqdm
import gc
import warnings
warnings.filterwarnings('ignore')

# Module DeepSeek MoE
from routing_deepseek import DeepSeekMoELayer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Đang chạy trên thiết bị: {device}")

def set_seed(seed):
    """Cố định seed để đảm bảo Reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f" Đã thiết lập Seed = {seed}")

c:\Users\Lenovo\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 Đang chạy trên thiết bị: cuda


In [ ]:
# ==========================================
# CELL 2: CẤU HÌNH (CONFIG) - ĐA ROUTING & XLM-R
# ==========================================
class Config:
    MODEL_NAME = "xlm-roberta-large" 
    TRAIN_CSV = r"D:\Study\NCKH_MoE_ Vietnamese_Natural_Language_Inference\Code\train.csv"
    VAL_CSV = r"D:\Study\NCKH_MoE_ Vietnamese_Natural_Language_Inference\Code\validation.csv"
    TEST_CSV = r"D:\Study\NCKH_MoE_ Vietnamese_Natural_Language_Inference\Code\test.csv"
    MAX_LEN = 256
    NUM_LABELS = 3
    
    # GRADIENT ACCUMULATION (Giải quyết OOM cho mô hình lớn)
    BATCH_SIZE = 4 
    ACCUMULATION_STEPS = 8 # Effective batch = 4 * 8 = 32
    LR = 1e-5
    EPOCHS = 10 
    PATIENCE = 3 
    SEEDS = [42, 123, 999] 
    
    NUM_EXPERTS = 32
    # Cấu hình danh sách chạy tuần tự 5 cơ chế
    ROUTING_TYPES_TO_TEST = ["expert_choice", "smoe", "micro", "adaptive", "deepseek"]
    CAPACITY_FACTOR = 1.5
    ROUTING_THRESHOLDS = 0.5
    
    CHECKPOINT_DIR = r"D:\Study\NCKH_MoE_ Vietnamese_Natural_Language_Inference\Code\checkpoints_xlmr_32"
    RESULTS_CSV = r"D:\Study\NCKH_MoE_ Vietnamese_Natural_Language_Inference\Code\results_xlmr_32exp.csv"

os.makedirs(Config.CHECKPOINT_DIR, exist_ok=True)

In [3]:
# ==========================================\n
# CELL 3: DATASET & DATALOADER (PRE-TOKENIZATION)
# ==========================================\n
label_map = {'entailment': 0, 'neutral': 1, 'contradiction': 2}
tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME)

class AdversarialNLIDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.labels = torch.tensor([label_map.get(str(l).strip().lower(), 1) for l in df['label']], dtype=torch.long)
        
        print(f" Đang pre-tokenize {len(df)} mẫu dữ liệu XLM-R... Vui lòng đợi.")
        self.encodings = tokenizer(
            df['premise'].astype(str).tolist(), 
            df['hypothesis'].astype(str).tolist(),
            add_special_tokens=True,
            max_length=max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        print(" Pre-tokenize hoàn tất!")
        
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': self.labels[idx]
        }

try:
    df_train = pd.read_csv(Config.TRAIN_CSV).dropna().reset_index(drop=True)
    df_val = pd.read_csv(Config.VAL_CSV).dropna().reset_index(drop=True)
    df_test = pd.read_csv(Config.TEST_CSV).dropna().reset_index(drop=True)

    train_loader = DataLoader(AdversarialNLIDataset(df_train, tokenizer, Config.MAX_LEN), batch_size=Config.BATCH_SIZE, shuffle=True, pin_memory=False)
    val_loader = DataLoader(AdversarialNLIDataset(df_val, tokenizer, Config.MAX_LEN), batch_size=Config.BATCH_SIZE, pin_memory=False)
    test_loader = DataLoader(AdversarialNLIDataset(df_test, tokenizer, Config.MAX_LEN), batch_size=Config.BATCH_SIZE, pin_memory=False)
except FileNotFoundError:
    print(" Không tìm thấy file CSV. Vui lòng kiểm tra lại.")

 Đang pre-tokenize 8012 mẫu dữ liệu XLM-R... Vui lòng đợi.
 Pre-tokenize hoàn tất!
 Đang pre-tokenize 1000 mẫu dữ liệu XLM-R... Vui lòng đợi.
 Pre-tokenize hoàn tất!
 Đang pre-tokenize 1000 mẫu dữ liệu XLM-R... Vui lòng đợi.
 Pre-tokenize hoàn tất!


In [ ]:
# ==========================================\n
# CELL 4: CHECKPOINT MANAGER & ENTROPY UTILS
# ==========================================\n
class CheckpointManager:
    def __init__(self, model, optimizer, scheduler, scaler, model_name="moe"):
        self.model = model
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.scaler = scaler
        self.model_name = model_name
        self.best_checkpoint_path = os.path.join(Config.CHECKPOINT_DIR, f"{model_name}_best.pth")
        self.last_checkpoint_path = os.path.join(Config.CHECKPOINT_DIR, f"{model_name}_last.pth")
        self.results_path = Config.RESULTS_CSV
        
        if not os.path.exists(self.results_path):
            df = pd.DataFrame(columns=["Seed", "Epoch", "Routing", "Val_Acc", "Val_F1", "GFlops", "Runtime_ms", "VRAM_MB", "Entropy"])
            df.to_csv(self.results_path, index=False)

    def save_checkpoint(self, epoch, val_f1, is_best=False):
        state = {
            'epoch': epoch, 
            'model_state': self.model.state_dict(),
            'best_val_f1': val_f1
        }
        try:
            torch.save(state, self.last_checkpoint_path)
            if is_best: torch.save(state, self.best_checkpoint_path)
        except Exception as e:
            print(f" Lỗi khi lưu checkpoint: {e}")

    def load_checkpoint(self):
        start_epoch, best_val_f1 = 0, 0.0
        if os.path.exists(self.last_checkpoint_path):
            state = torch.load(self.last_checkpoint_path)
            model_state = state['model_state']
            clean_state_dict = {k: v for k, v in model_state.items() if 'total_ops' not in k and 'total_params' not in k}
            self.model.load_state_dict(clean_state_dict, strict=False)
            start_epoch = state['epoch'] + 1
            best_val_f1 = state.get('best_val_f1', 0.0)
            print(f" Đã khôi phục thành công! Bắt đầu từ Epoch {start_epoch + 1}")
        return start_epoch, best_val_f1

    def log_results(self, row):
        df = pd.read_csv(self.results_path)
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
        df.to_csv(self.results_path, index=False)

def calculate_routing_metrics(model):
    """Tính toán Routing Entropy và xuất tần suất sử dụng của chuyên gia"""
    metrics = {
        "entropy": 0.0,
        "expert_usage_distribution": None # Dùng mảng này để vẽ Pareto sau khi train
    }
    
    try:
        moe = model.moe_layer
        if hasattr(moe, "gate_logits") and moe.gate_logits is not None:
            probs = torch.softmax(moe.gate_logits, dim=-1)
            entropy = -(probs * torch.log(probs + 1e-9)).sum(dim=-1).mean()
            
            # Lấy xác suất trung bình phân bổ cho từng expert
            expert_usage = probs.mean(dim=0).cpu().numpy().tolist() 
            metrics["entropy"] = entropy.item()
            metrics["expert_usage_distribution"] = expert_usage
            
        elif hasattr(moe, "expert_usage") and moe.expert_usage is not None:
            usage = moe.expert_usage.float()
            probs = usage / (usage.sum() + 1e-9)
            entropy = -(probs * torch.log(probs + 1e-9)).sum()
            
            metrics["entropy"] = entropy.item()
            metrics["expert_usage_distribution"] = usage.cpu().numpy().tolist()
    except Exception:
        pass
        
    return metrics

In [5]:
# ==========================================\n
# CELL 5: MODEL ARCHITECTURE (XLM-R)
# ==========================================\n
class LayerAttentionPooling(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, hidden_size), 
            nn.Tanh(), 
            nn.Linear(hidden_size, 1)
        )
        
    def forward(self, hidden_states, attention_mask):
        attn_weights = self.attention(hidden_states).squeeze(-1)
        min_val = torch.finfo(attn_weights.dtype).min 
        attn_weights = attn_weights.masked_fill(attention_mask == 0, min_val)
        attn_weights = F.softmax(attn_weights, dim=-1)
        return torch.bmm(attn_weights.unsqueeze(1), hidden_states).squeeze(1)

class UnifiedMoENLI(nn.Module):
    def __init__(self, config, routing_type="expert_choice"):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(config.MODEL_NAME)
        hidden_size = self.backbone.config.hidden_size
        
        # Đóng băng 12 layer đầu
        for name, param in self.backbone.named_parameters():
            if 'encoder.layer' in name and int(name.split('.')[2]) < 12:
                param.requires_grad = False

        # Khởi tạo linh hoạt, loại bỏ hardcode fallback
        if routing_type == "smoe": self.moe_layer = SMoELayer(hidden_size, config.NUM_EXPERTS)
        elif routing_type == "micro": self.moe_layer = MICROMoELayer(hidden_size)
        elif routing_type == "expert_choice": self.moe_layer = ExpertChoiceMoELayer(hidden_size, config.NUM_EXPERTS, config.CAPACITY_FACTOR)
        elif routing_type == "adaptive": self.moe_layer = AdaptiveDynamicMoELayer(hidden_size, config.NUM_EXPERTS, getattr(config, 'ROUTING_THRESHOLDS', 0.5))
        elif routing_type == "deepseek": self.moe_layer = DeepSeekMoELayer(hidden_size, num_shared_experts=2, num_routed_experts=config.NUM_EXPERTS-2)
        else: raise ValueError(f" Routing type '{routing_type}' không hợp lệ!")
            
        self.attention_pooling = LayerAttentionPooling(hidden_size)
        self.classifier = nn.Sequential(
            nn.Dropout(0.2), 
            nn.Linear(hidden_size, hidden_size // 2), 
            nn.GELU(), 
            nn.Linear(hidden_size // 2, config.NUM_LABELS)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        moe_output = self.moe_layer(outputs.last_hidden_state)
        
        # Xử lý trường hợp MoE layer trả về tuple (output, aux_loss)
        aux_loss = 0.0
        if isinstance(moe_output, tuple):
            moe_output, aux_loss = moe_output
            
        pooled_output = self.attention_pooling(moe_output, attention_mask)
        logits = self.classifier(pooled_output)
        
        return logits, aux_loss

In [ ]:
# ==========================================
# CELL 6: TRAINING LOOP (ĐA ROUTING + ACCUMULATION CHUẨN)
# ==========================================
import math

all_test_results = []

for seed in Config.SEEDS:
    set_seed(seed)
    
    for current_routing in Config.ROUTING_TYPES_TO_TEST:
        print(f"\n{'='*70}")
        print(f"🚀 SEED {seed} | XLM-R | ROUTING: {current_routing.upper()} (32 EXPERTS)")
        print(f"{'='*70}")
        
        model = UnifiedMoENLI(Config(), routing_type=current_routing).to(device)
        optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=Config.LR, weight_decay=0.01)
        
        # Tính toán tổng số bước chính xác khi có Accumulation
        total_steps = math.ceil(len(train_loader) / Config.ACCUMULATION_STEPS) * Config.EPOCHS
        scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)
        
        scaler = GradScaler()
        criterion = nn.CrossEntropyLoss()
        
        # Đặt tên checkpoint riêng biệt theo từng cơ chế định tuyến để tính năng resume hoạt động chính xác
        model_name = f"xlmr_{current_routing}_32exp_seed{seed}"
        checkpoint_manager = CheckpointManager(model, optimizer, scheduler, scaler, model_name=model_name)
        start_epoch, best_val_f1 = checkpoint_manager.load_checkpoint()

        dummy_ids = torch.ones(1, Config.MAX_LEN, dtype=torch.long).to(device)
        dummy_mask = torch.ones(1, Config.MAX_LEN, dtype=torch.long).to(device)
        macs, _ = profile(model, inputs=(dummy_ids, dummy_mask), verbose=False)
        gflops = (macs * 2) / 1e9
        print(f"📊 Tài nguyên ước tính: {gflops:.2f} GFlops")

        final_gflops, final_runtime, final_vram, final_entropy = gflops, 0, 0, 0
        early_stop_counter = 0

        for epoch in range(start_epoch, Config.EPOCHS):
            model.train()
            optimizer.zero_grad(set_to_none=True) 
            train_iterator = tqdm(train_loader, desc=f"[{current_routing.upper()}] Ep {epoch+1}/{Config.EPOCHS} [Train]", leave=False)
            
            for step, batch in enumerate(train_iterator):
                ids = batch['input_ids'].to(device, non_blocking=True)
                mask = batch['attention_mask'].to(device, non_blocking=True)
                labels = batch['labels'].to(device, non_blocking=True)
                
                with autocast():
                    logits, aux_loss = model(ids, mask)
                    main_loss = criterion(logits, labels)
                    loss = (main_loss + aux_loss) / Config.ACCUMULATION_STEPS
                    
                scaler.scale(loss).backward()
                
                if (step + 1) % Config.ACCUMULATION_STEPS == 0 or (step + 1) == len(train_loader):
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                    scheduler.step()
                    optimizer.zero_grad(set_to_none=True) 
                    
                train_iterator.set_postfix(loss=f"{(loss.item() * Config.ACCUMULATION_STEPS):.4f}")

            # --- VALIDATION ---
            model.eval()
            val_preds, val_labels = [], []
            torch.cuda.synchronize()
            start_time = time.time()
            
            with torch.inference_mode():
                val_iterator = tqdm(val_loader, desc=f"[{current_routing.upper()}] Ep {epoch+1}/{Config.EPOCHS} [Val]", leave=False)
                for batch in val_iterator:
                    b_ids = batch['input_ids'].to(device, non_blocking=True)
                    b_mask = batch['attention_mask'].to(device, non_blocking=True)
                    b_labels = batch['labels'].to(device, non_blocking=True)
                    
                    with autocast():
                        logits, _ = model(b_ids, b_mask)
                    
                    val_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
                    val_labels.extend(b_labels.cpu().numpy())
                    
            torch.cuda.synchronize()
            runtime_ms = ((time.time() - start_time) / len(val_loader)) * 1000
            vram_mb = torch.cuda.max_memory_allocated() / (1024 * 1024)
            val_acc = accuracy_score(val_labels, val_preds)
            val_f1 = f1_score(val_labels, val_preds, average='macro')
            routing_stats = calculate_routing_metrics(model)
            entropy_val = routing_stats["entropy"]
            expert_distribution = routing_stats["expert_usage_distribution"]
            
            print(f"[{current_routing.upper()}] Ep {epoch+1} | Acc: {val_acc:.4f} | F1: {val_f1:.4f} | {runtime_ms:.2f} ms/b | VRAM: {vram_mb:.0f} MB")
            
            # Ghi nhận kết quả vào file CSV sau mỗi Epoch
            checkpoint_manager.log_results({
                "Seed": seed, "Epoch": epoch+1, "Routing": current_routing, 
                "Val_Acc": val_acc, "Val_F1": val_f1, "GFlops": gflops, 
                "Runtime_ms": runtime_ms, "VRAM_MB": vram_mb, "Entropy": entropy_val,
                "Expert_Usage": str(expert_distribution)
            })
            
            is_best = val_f1 > best_val_f1
            if is_best: 
                best_val_f1 = val_f1
                final_runtime, final_vram, final_entropy = runtime_ms, vram_mb, entropy_val
                early_stop_counter = 0 
                print("✨ Validation F1 cải thiện, lưu Best Checkpoint.")
            else:
                early_stop_counter += 1
                print(f"⚠️ Validation F1 không tăng. Early Stop: {early_stop_counter}/{Config.PATIENCE}")
                
            checkpoint_manager.save_checkpoint(epoch, best_val_f1, is_best)
            if early_stop_counter >= Config.PATIENCE:
                print(f"🛑 Kích hoạt Early Stopping tại Epoch {epoch+1}!")
                break

        # --- TEST TẬP CHUẨN ---
        print(f"\n📥 Loading best checkpoint cho {current_routing.upper()} (Seed {seed})...")
        try:
            state = torch.load(checkpoint_manager.best_checkpoint_path, map_location=device)
            model.load_state_dict(state["model_state"], strict=False)
            model.eval()
            
            test_preds, test_labels = [], []
            with torch.inference_mode():
                for batch in test_loader:
                    ids = batch["input_ids"].to(device, non_blocking=True)
                    mask = batch["attention_mask"].to(device, non_blocking=True)
                    labels = batch["labels"].to(device, non_blocking=True)
                    
                    with autocast():
                        # Đã sửa: Nhận đúng cấu trúc tuple (logits, aux_loss) để tránh sập luồng tính toán
                        logits, _ = model(ids, mask)
                    test_preds.extend(torch.argmax(logits, 1).cpu().numpy())
                    test_labels.extend(labels.cpu().numpy())

            test_acc = accuracy_score(test_labels, test_preds)
            test_f1 = f1_score(test_labels, test_preds, average="macro")
            print(f"🎯 TEST ACC = {test_acc:.4f} | TEST F1 = {test_f1:.4f}")
            
            all_test_results.append({
                "Seed": seed, "Routing": current_routing, "Test_ACC": test_acc, "Test_F1": test_f1,
                "GFLOPS": final_gflops, "Runtime": final_runtime, "VRAM": final_vram, "Entropy": final_entropy
            })
        except Exception as e:
            print(f"❌ Không thể chạy Test. Lỗi: {e}")
        
        # Dọn dẹp GPU bộ nhớ trước khi chuyển sang cơ chế định tuyến tiếp theo
        del model, optimizer, scheduler, checkpoint_manager
        torch.cuda.empty_cache()
        gc.collect()

print("✅ Hoàn tất huấn luyện toàn bộ các kiến trúc định tuyến!")
pd.DataFrame(all_test_results).to_csv(Config.RESULTS_CSV.replace(".csv", "_test_summary.csv"), index=False)

 Đã thiết lập Seed = 42

🚀 SEED 42 | XLM-R | ROUTING: DEEPSEEK (32 EXPERTS)
📊 Tài nguyên ước tính: 172.46 GFlops


KeyboardInterrupt: 